In [ ]:
import os
import re
import json
from datetime import datetime
import random
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

# 1. Test Dataset Generation from Documents

In [ ]:
from evaluation.eval import prepare_testset_documents

In [ ]:
docs = prepare_testset_documents("eval_data")

In [6]:
from rag_pipeline.query_engine import vectorstore

In [ ]:
vectorstore_db = vectorstore(documents=docs, batch_size=250)

Adding 144289 documents to PGVector...


# 2. Synthetic QA Dataset Generation

In [7]:
# imports for synthetic QA Dataset Generation
from ragas import RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama                     # ollama can run locally
from langchain_huggingface import HuggingFaceEmbeddings

In [8]:
# csv files are temporarily filtered out because they are large and dominate the dataset
gen_docs = [file for file in docs if not file.metadata['filename'].endswith('.csv')]

# subsample documents to control cost and runtime during synthetic QA generation
random_docs = random.sample(gen_docs, 200)

run_config = RunConfig(max_workers=2, timeout=180)

In [ ]:
generator_llm = LangchainLLMWrapper(
                ChatOllama(
                    model="qwen2.5",
                    temperature=0,
                    # model_kwargs={"response_format":{"type":"json_object"}}
                    format="json"
                    )
                )

generator_embeddings = LangchainEmbeddingsWrapper(
                HuggingFaceEmbeddings(
                    model_name="sentence-transformers/all-MiniLM-L6-v2"
                    )
                )           

In [ ]:
testset = generate_qa_dataset(
                                random_docs, generator_llm, generator_embeddings,  
                                run_config=run_config,
                                testset_size=100
                            )

In [ ]:
df = testset.to_pandas()

In [ ]:
testset.to_pandas().to_json("test_data/rag_evaluation_testset.json", orient="records", indent=2)

# 3. RAG Pipeline Execution

In [ ]:
from evaluation.eval import generate_rag_responses
from rag_pipeline.query_engine import vectorstore

In [ ]:
df = pd.read_json("test_data/rag_evaluation_testset.json")

In [ ]:
vectorstore_db = vectorstore('chroma_db')

In [ ]:
df_32 = pd.read_json("test_data/curated_rag_evaluation_testset_32.jsonl", orient="records", lines=True)

In [ ]:
generate_rag_responses(df_32, vectorstore_db, session_id="test_session")

In [ ]:
with open("test_data/rag_results_v5.jsonl", "r") as f:
    rag_results = [json.loads(line) for line in f]

In [ ]:
# print(rag_results[-1]['user_input'])
print(df.iloc[31].user_input)

# 4. RAG Evaluation

In [ ]:
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    ResponseRelevancy,
)
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
evaluator_llm = LangchainLLMWrapper(
                    ChatGoogleGenerativeAI(
                        model="gemini-flash-lite-latest",
                        temperature=0,
                        # model_kwargs={"response_format": {"type": "json_object"}},
                    )
                )
evaluator_embeddings = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        )

In [ ]:
metrics = [
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
]

run_config = RunConfig(max_workers=1, timeout=600)

In [ ]:
from evaluation.eval import run_batch_evaluation

In [ ]:
file_path = "evaluation/rag_eval_v5.jsonl"

In [ ]:
run_batch_evaluation(rag_results, metrics, run_config, file_path)

In [ ]:
scores = pd.read_json("evaluation/rag_eval_v5.jsonl", lines=True)
print(scores[["llm_context_precision_with_reference","context_recall",
              "faithfulness","answer_relevancy"]].mean())

In [ ]:
from evaluation.eval import get_score

In [ ]:
get_score("evaluation/rag_eval_v5.csv")

In [ ]:
# scores.to_csv("/Volumes/LaCie/Projects_portfolio/IntelliQA/evaluation/rag_eval_v5.csv")

In [ ]:
scores[scores['context_recall']==0]

In [ ]:
result_df = pd.read_csv("evaluation/rag_eval_v5.csv")

In [ ]:
recall_fails = result_df[result_df['context_recall']==0][['user_input','retrieved_contexts','response', 'reference']]

In [ ]:
print(result_df['user_input'].iloc[1])

In [ ]:
print(result_df['retrieved_contexts'].iloc[3])

In [ ]:
from IPython.display import display, HTML

subset = scores[scores['context_recall'] == 0][
    ['user_input', 'retrieved_contexts', 'response', 'reference']
]

display(HTML(subset.to_html().replace('<td>', '<td style="text-align:left; white-space:pre-wrap; word-wrap:break-word; max-width:400px;">')))

In [ ]:
scores